# Feature Reduction Notebook (Top-20 → PCA / Autoencoder)
Loads engineered **wide** features/targets, builds a clean `(Date, Ticker)` panel for your **Top-20** features, then creates:
- **Dataset A:** Raw Top-20 (impute + standardize)
- **Dataset B:** PCA-reduced (retain 90–95% variance)
- **Dataset C:** Autoencoder-compressed (latent dims)

Also trains **Ridge / ElasticNet** for a fast sanity check and saves Parquet datasets for ML/DL.


In [16]:
# %% [code] 1) Imports & Config
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import spearmanr

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# --- Paths ---
PROJECT_DIR = Path(".")
DATA_DIR    = PROJECT_DIR

FEATURES_WIDE_PATH = DATA_DIR / "features_10y_100tickers_wide.parquet"
TARGETS_WIDE_PATH  = DATA_DIR / "targets_10y_100tickers_wide.parquet"

OUTDIR = PROJECT_DIR / "reduction_outputs_10y_100t"
OUTDIR.mkdir(parents=True, exist_ok=True)

# --- Repro ---
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# --- Split / Reduction params ---
TEST_FRACTION = 0.2

PCA_VAR = 0.90          # keep 90% variance (change to 0.95 if you want)

AE_LATENT   = 8
AE_EPOCHS   = 40
AE_PATIENCE = 6
AE_BATCH    = 4096
AE_LR       = 1e-3
AE_WD       = 1e-4

# --- Your Top-20 list ---
TOP20 = [
    "semi_variance_63d","volatility_21d","downside_volatility_63d","kurtosis_63d",
    "volatility_126d","volatility_63d","turnover","cross_sectional_rank_volatility",
    "volume_volatility_21d","volume_volatility_63d","rolling_corr_market_63d",
    "r2_FF_12m","r2_FF_6m","beta_SMB_6m","beta_SMB_12m","skewness_63d",
    "pca_component_1","autocorr_63d","ma_ratio_63_126","drawdown_63d"
]

TARGET_PREFIX = "target_return_next1m__"


Device: cpu


In [18]:
# %% [code] 2) Load wide features/targets (correct directory)

FEATURE_DIR = Path(r"D:\Thesis\Old\Codes\New\feature_engineering_outputs")

FEATURES_WIDE_PATH = FEATURE_DIR / "features_10y_100tickers_wide.parquet"
TARGETS_WIDE_PATH  = FEATURE_DIR / "targets_10y_100tickers_wide.parquet"

assert FEATURES_WIDE_PATH.exists(), f"Missing: {FEATURES_WIDE_PATH}"
assert TARGETS_WIDE_PATH.exists(),  f"Missing: {TARGETS_WIDE_PATH}"

Features = pd.read_parquet(FEATURES_WIDE_PATH)
Targets  = pd.read_parquet(TARGETS_WIDE_PATH)

# ensure datetime index
Features.index = pd.to_datetime(Features.index)
Targets.index  = pd.to_datetime(Targets.index)

# drop duplicate columns if any
Features = Features.loc[:, ~Features.columns.duplicated(keep="first")]
Targets  = Targets.loc[:,  ~Targets.columns.duplicated(keep="first")]

print("Loaded:")
print("  Features:", Features.shape)
print("  Targets :", Targets.shape)
print("Date range:", Features.index.min().date(), "→", Features.index.max().date())

# target columns
target_cols = [c for c in Targets.columns if c.startswith(TARGET_PREFIX)]
print("Target columns found:", len(target_cols))
print("Example target cols:", target_cols[:5])

# clean infinities
Features = Features.replace([np.inf, -np.inf], np.nan)
Targets  = Targets.replace([np.inf, -np.inf], np.nan)


Loaded:
  Features: (2515, 5100)
  Targets : (2515, 200)
Date range: 2015-06-30 → 2025-06-30
Target columns found: 100
Example target cols: ['target_return_next1m__AAPL', 'target_return_next1m__AMP', 'target_return_next1m__AMT', 'target_return_next1m__AON', 'target_return_next1m__APA']


In [19]:
# %% [code] 3) Universe (tickers) + resolve Top-20 features that exist in parquet

def extract_tickers_from_wide(cols):
    tickers = set()
    for c in cols:
        if "__" in c:
            tickers.add(c.split("__", 1)[1])
    return sorted(tickers)

def available_base_features(cols):
    return sorted({c.split("__", 1)[0] for c in cols if "__" in c})

tickers = extract_tickers_from_wide(Features.columns)
bases   = set(available_base_features(Features.columns))

print("Tickers detected in Features:", len(tickers))
print("First 10 tickers:", tickers[:10])
print("Base features detected:", len(bases))

# Aliases to handle naming differences (edit here if needed)
FEATURE_ALIASES = {
    "rolling_corr_market_63d": ["rolling_corr_sp500_63d", "rolling_corr_market_126d"],
}

def resolve_feature_name(feat):
    if feat in bases:
        return feat
    for cand in FEATURE_ALIASES.get(feat, []):
        if cand in bases:
            return cand
    return None

TOP20_RESOLVED = []
TOP20_MISSING  = []
for f in TOP20:
    r = resolve_feature_name(f)
    if r is None:
        TOP20_MISSING.append(f)
    else:
        TOP20_RESOLVED.append(r)

print("\n✅ Features we will use:", TOP20_RESOLVED)
print("Count used:", len(TOP20_RESOLVED))

if TOP20_MISSING:
    print("\n⚠️ Missing (not in parquet):", TOP20_MISSING)

# Also confirm targets tickers (should match)
target_tickers = sorted({c.split("__",1)[1] for c in Targets.columns if c.startswith(TARGET_PREFIX)})
print("\nTickers in Targets (prefix):", len(target_tickers))
print("Targets tickers match Features tickers?", set(target_tickers) == set(tickers))


Tickers detected in Features: 180
First 10 tickers: ['AAPL', 'AMP', 'AMT', 'AON', 'APA', 'APTV', 'ATO', 'AVGO', 'AXON', 'BMY']
Base features detected: 50

✅ Features we will use: ['semi_variance_63d', 'volatility_21d', 'downside_volatility_63d', 'kurtosis_63d', 'volatility_126d', 'volatility_63d', 'turnover', 'cross_sectional_rank_volatility', 'volume_volatility_21d', 'volume_volatility_63d', 'rolling_corr_market_63d', 'r2_FF_12m', 'r2_FF_6m', 'beta_SMB_6m', 'beta_SMB_12m', 'skewness_63d', 'pca_component_1', 'autocorr_63d', 'ma_ratio_63_126', 'drawdown_63d']
Count used: 20

Tickers in Targets (prefix): 100
Targets tickers match Features tickers? False


In [20]:
# %% [code] 4) Build a clean (Date,Ticker) panel using ONLY tickers that exist in Targets

# use the targets universe as the truth (you said these are "100 tickers")
tickers_use = target_tickers.copy()

print("Using tickers:", len(tickers_use))
print("First 10:", tickers_use[:10])

def build_panel_top_features(Features_wide, Targets_wide, tickers, features_list, target_prefix):
    # --- target wide -> long ---
    y_cols = [f"{target_prefix}{t}" for t in tickers if f"{target_prefix}{t}" in Targets_wide.columns]
    if len(y_cols) != len(tickers):
        missing_y = [t for t in tickers if f"{target_prefix}{t}" not in Targets_wide.columns]
        raise ValueError(f"Targets missing for {len(missing_y)} tickers (e.g. {missing_y[:10]})")

    Y = Targets_wide[y_cols].copy()
    Y.columns = [c.split("__", 1)[1] for c in Y.columns]
    Y = Y.reindex(columns=tickers)
    y_long = Y.stack().rename("y").reset_index()
    y_long.columns = ["Date", "Ticker", "y"]
    y_long["Date"] = pd.to_datetime(y_long["Date"])

    # --- features wide -> long (merge one by one to avoid huge concat issues) ---
    panel = y_long

    for feat in tqdm(features_list, desc="Merging features"):
        cols = [f"{feat}__{t}" for t in tickers if f"{feat}__{t}" in Features_wide.columns]
        if len(cols) != len(tickers):
            missing_x = [t for t in tickers if f"{feat}__{t}" not in Features_wide.columns]
            raise ValueError(f"Feature '{feat}' missing for {len(missing_x)} tickers (e.g. {missing_x[:10]})")

        X = Features_wide[cols].copy()
        X.columns = [c.split("__", 1)[1] for c in X.columns]
        X = X.reindex(columns=tickers)

        x_long = X.stack().rename(feat).reset_index()
        x_long.columns = ["Date", "Ticker", feat]
        x_long["Date"] = pd.to_datetime(x_long["Date"])

        panel = panel.merge(x_long, on=["Date", "Ticker"], how="left")

    panel = panel.sort_values(["Date", "Ticker"]).set_index(["Date", "Ticker"]).sort_index()
    panel = panel.replace([np.inf, -np.inf], np.nan)
    panel = panel.dropna(subset=["y"])
    return panel

panel = build_panel_top_features(Features, Targets, tickers_use, TOP20_RESOLVED, TARGET_PREFIX)

print("✅ Panel shape:", panel.shape)
print("Dates:", panel.index.get_level_values("Date").nunique(),
      "| Tickers:", panel.index.get_level_values("Ticker").nunique())

display(panel.head())


Using tickers: 100
First 10: ['AAPL', 'AMP', 'AMT', 'AON', 'APA', 'APTV', 'ATO', 'AVGO', 'AXON', 'BMY']


Merging features:   0%|          | 0/20 [00:00<?, ?it/s]

✅ Panel shape: (249400, 21)
Dates: 2494 | Tickers: 100


y  semi_variance_63d  volatility_21d  \
Date       Ticker                                                
2015-06-30 AAPL   -0.024396                NaN             NaN   
           AMP     0.018593                NaN             NaN   
           AMT     0.017794                NaN             NaN   
           AON     0.025693                NaN             NaN   
           APA    -0.189474                NaN             NaN   

                   downside_volatility_63d  kurtosis_63d  volatility_126d  \
Date       Ticker                                                           
2015-06-30 AAPL                        NaN           NaN              NaN   
           AMP                         NaN           NaN              NaN   
           AMT                         NaN           NaN              NaN   
           AON                         NaN           NaN              NaN   
           APA                         NaN           NaN              NaN   

                   volatility_63d  turnover  cross_sectional_rank_volatility  \
Date       Ticker                                                              
2015-06-30 AAPL               NaN       NaN                              NaN   
           AMP                NaN       NaN                              NaN   
           AMT                NaN       NaN                              NaN   
           AON                NaN       NaN                              NaN   
           APA                NaN       NaN                              NaN   

                   volume_volatility_21d  ...  rolling_corr_market_63d  \
Date       Ticker                         ...                            
2015-06-30 AAPL                      NaN  ...                      NaN   
           AMP                       NaN  ...                      NaN   
           AMT                       NaN  ...                      NaN   
           AON                       NaN  ...                      NaN   
           APA                       NaN  ...                      NaN   

                   r2_FF_12m  r2_FF_6m  beta_SMB_6m  beta_SMB_12m  \
Date       Ticker                                                   
2015-06-30 AAPL          NaN       NaN          NaN           NaN   
           AMP           NaN       NaN          NaN           NaN   
           AMT           NaN       NaN          NaN           NaN   
           AON           NaN       NaN          NaN           NaN   
           APA           NaN       NaN          NaN           NaN   

                   skewness_63d  pca_component_1  autocorr_63d  \
Date       Ticker                                                
2015-06-30 AAPL             NaN              NaN           NaN   
           AMP              NaN              NaN           NaN   
           AMT              NaN              NaN           NaN   
           AON              NaN              NaN           NaN   
           APA              NaN              NaN           NaN   

                   ma_ratio_63_126  drawdown_63d  
Date       Ticker                                 
2015-06-30 AAPL                NaN           NaN  
           AMP                 NaN           NaN  
           AMT                 NaN           NaN  
           AON                 NaN           NaN  
           APA                 NaN           NaN  

[5 rows x 21 columns]

In [21]:
# %% [code] 5) Time split + preprocessing (impute + scale)

FEATURE_COLS = TOP20_RESOLVED
TARGET_COL = "y"

# sort index explicitly
panel = panel.sort_index()

dates = panel.index.get_level_values("Date").unique().sort_values()
split_idx = int(len(dates) * (1 - TEST_FRACTION))
train_dates = dates[:split_idx]
test_dates  = dates[split_idx:]

train = panel.loc[train_dates]
test  = panel.loc[test_dates]

print("Train dates:", train_dates.min().date(), "→", train_dates.max().date())
print("Test  dates:", test_dates.min().date(),  "→", test_dates.max().date())
print("Train rows:", train.shape[0], "| Test rows:", test.shape[0])

X_train = train[FEATURE_COLS]
X_test  = test[FEATURE_COLS]

y_train = train[TARGET_COL]
y_test  = test[TARGET_COL]

# --- Impute (fit on train only) ---
imputer = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(
    imputer.fit_transform(X_train),
    index=X_train.index,
    columns=FEATURE_COLS
)
X_test_imp = pd.DataFrame(
    imputer.transform(X_test),
    index=X_test.index,
    columns=FEATURE_COLS
)

# --- Standardize (fit on train only) ---
scaler = StandardScaler()
X_train_std = pd.DataFrame(
    scaler.fit_transform(X_train_imp),
    index=X_train_imp.index,
    columns=FEATURE_COLS
)
X_test_std = pd.DataFrame(
    scaler.transform(X_test_imp),
    index=X_test_imp.index,
    columns=FEATURE_COLS
)

print("✅ Preprocessing done")
print("NaNs train:", X_train_std.isna().sum().sum(),
      "| test:", X_test_std.isna().sum().sum())


Train dates: 2015-06-30 → 2023-06-01
Test  dates: 2023-06-02 → 2025-05-29
Train rows: 199500 | Test rows: 49900
✅ Preprocessing done
NaNs train: 0 | test: 0


In [22]:
# %% [code] 6.1) Fit PCA on training data only

pca = PCA(n_components=PCA_VAR, random_state=RANDOM_SEED)
X_train_pca = pca.fit_transform(X_train_std)
X_test_pca  = pca.transform(X_test_std)

print("Original feature dim :", X_train_std.shape[1])
print("PCA components kept  :", X_train_pca.shape[1])
print("Explained variance   :", round(pca.explained_variance_ratio_.sum(), 4))

# Scree-style table
pca_table = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(len(pca.explained_variance_ratio_))],
    "Explained_Var": pca.explained_variance_ratio_,
    "Cumulative": np.cumsum(pca.explained_variance_ratio_)
})
display(pca_table.head(10))


Original feature dim : 20
PCA components kept  : 11
Explained variance   : 0.9149


,PC,Explained_Var,Cumulative
0,PC1,0.307934,0.307934
1,PC2,0.119357,0.427291
2,PC3,0.107835,0.535126
3,PC4,0.069883,0.605009
4,PC5,0.062583,0.667592
5,PC6,0.053253,0.720845
6,PC7,0.049955,0.770799
7,PC8,0.048369,0.819168
8,PC9,0.037341,0.856509
9,PC10,0.035086,0.891595


In [23]:
# %% [code] 6.2) PCA loadings (what each component represents)

loadings = pd.DataFrame(
    pca.components_.T,
    index=FEATURE_COLS,
    columns=[f"PC{i+1}" for i in range(X_train_pca.shape[1])]
)

display(loadings.abs().sort_values("PC1", ascending=False).head(10))


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11
volatility_63d,0.377147,0.131357,0.077022,0.002442,0.119076,0.084827,0.006393,0.002581,0.029277,0.107427,0.030310
downside_volatility_63d,0.365499,0.171572,0.148266,0.035723,0.054494,0.020270,0.023305,0.097711,0.122858,0.091931,0.085169
volatility_126d,0.354365,0.051687,0.015546,0.027362,0.142100,0.007182,0.008874,0.025089,0.080885,0.199351,0.234744
volatility_21d,0.332027,0.128785,0.072535,0.040793,0.101172,0.085428,0.008337,0.043420,0.014630,0.250815,0.128939
drawdown_63d,0.324376,0.061542,0.040684,0.149998,0.127872,0.017918,0.026374,0.112061,0.185249,0.152093,0.439475
semi_variance_63d,0.319705,0.167102,0.202452,0.010448,0.023092,0.030652,0.009327,0.092897,0.009582,0.104590,0.513181
cross_sectional_rank_volatility,0.280101,0.023054,0.286321,0.071746,0.004235,0.108160,0.018523,0.005500,0.244016,0.070901,0.518055
r2_FF_12m,0.219456,0.293797,0.108402,0.026089,0.199664,0.515463,0.042296,0.092344,0.067520,0.072186,0.037572
r2_FF_6m,0.213501,0.284220,0.067350,0.000754,0.188197,0.549894,0.038256,0.094036,0.075942,0.161480,0.148246
rolling_corr_market_63d,0.160816,0.382312,0.217000,0.398252,0.052759,0.267394,0.027142,0.155180,0.064121,0.003128,0.031492


In [24]:
# %% [code] 6.3) Ridge & ElasticNet on PCA features

results_pca = []

for model_name, model in {
    "Ridge": Ridge(alpha=1.0),
    "ElasticNet": ElasticNet(alpha=0.01, l1_ratio=0.5)
}.items():
    model.fit(X_train_pca, y_train.values)
    y_pred = model.predict(X_test_pca)

    r2 = r2_score(y_test.values, y_pred)
    mse = mean_squared_error(y_test.values, y_pred)

    ic = spearmanr(y_test.values, y_pred).correlation

    results_pca.append({
        "model": model_name,
        "OOS_R2": r2,
        "MSE": mse,
        "IC": ic
    })

results_pca = pd.DataFrame(results_pca)
display(results_pca)


,model,OOS_R2,MSE,IC
0,Ridge,0.009943,0.007341,0.054850
1,ElasticNet,0.007010,0.007363,0.042626


In [25]:
# %% [code] 7.1) Autoencoder datasets + dataloaders (CPU-friendly)

LATENT_DIM = 8
BATCH_SIZE = 2048
EPOCHS = 40
LR = 1e-3
WEIGHT_DECAY = 1e-5

# tensors
Xtr = torch.tensor(X_train_std.values, dtype=torch.float32)
Xte = torch.tensor(X_test_std.values, dtype=torch.float32)

train_ds = TensorDataset(Xtr)
test_ds  = TensorDataset(Xte)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

print("X_train tensor:", Xtr.shape, "| X_test tensor:", Xte.shape)


X_train tensor: torch.Size([199500, 20]) | X_test tensor: torch.Size([49900, 20])


In [26]:
# %% [code] 7.2) Define Autoencoder (MLP) for tabular features

class TabularAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=8):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        xhat = self.decoder(z)
        return xhat, z

ae = TabularAutoencoder(input_dim=Xtr.shape[1], latent_dim=LATENT_DIM).to(DEVICE)
print(ae)


TabularAutoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=20, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.1, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): ReLU()
    (6): Linear(in_features=32, out_features=8, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=8, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=20, bias=True)
  )
)


In [27]:
# %% [code] 7.3) Train autoencoder (reconstruction loss)

criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(ae.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

best_val = np.inf
best_state = None
history = []

for epoch in range(1, EPOCHS + 1):
    ae.train()
    tr_loss = 0.0
    for (xb,) in tqdm(train_loader, desc=f"AE Train {epoch:02d}/{EPOCHS}", leave=False):
        xb = xb.to(DEVICE)
        optimizer.zero_grad()
        xhat, _ = ae(xb)
        loss = criterion(xhat, xb)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item() * xb.size(0)
    tr_loss /= len(train_ds)

    ae.eval()
    va_loss = 0.0
    with torch.no_grad():
        for (xb,) in test_loader:
            xb = xb.to(DEVICE)
            xhat, _ = ae(xb)
            loss = criterion(xhat, xb)
            va_loss += loss.item() * xb.size(0)
    va_loss /= len(test_ds)

    history.append((epoch, tr_loss, va_loss))
    if va_loss < best_val:
        best_val = va_loss
        best_state = {k: v.cpu().clone() for k, v in ae.state_dict().items()}

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d} | train MSE {tr_loss:.6f} | test MSE {va_loss:.6f} | best {best_val:.6f}")

# load best
ae.load_state_dict(best_state)
print("✅ AE trained. Best test reconstruction MSE:", best_val)


AE Train 01/40:   0%|          | 0/98 [00:00<?, ?it/s]

Epoch 01 | train MSE 0.664748 | test MSE 0.400932 | best 0.400932


AE Train 02/40:   0%|          | 0/98 [00:00<?, ?it/s]

AE Train 03/40:   0%|          | 0/98 [00:00<?, ?it/s]

AE Train 04/40:   0%|          | 0/98 [00:00<?, ?it/s]

...

AE Train 37/40:   0%|          | 0/98 [00:00<?, ?it/s]

AE Train 38/40:   0%|          | 0/98 [00:00<?, ?it/s]

AE Train 39/40:   0%|          | 0/98 [00:00<?, ?it/s]

AE Train 40/40:   0%|          | 0/98 [00:00<?, ?it/s]

Epoch 40 | train MSE 0.098672 | test MSE 0.096584 | best 0.095152
✅ AE trained. Best test reconstruction MSE: 0.09515231524058478


In [28]:
# %% [code] 7.4) Extract AE latent features and evaluate (Ridge/ElasticNet)

ae.eval()
with torch.no_grad():
    _, Ztr = ae(Xtr.to(DEVICE))
    _, Zte = ae(Xte.to(DEVICE))

Ztr = Ztr.cpu().numpy()
Zte = Zte.cpu().numpy()

print("Latent train shape:", Ztr.shape, "| Latent test shape:", Zte.shape)

results_ae = []
for model_name, model in {
    "Ridge": Ridge(alpha=1.0),
    "ElasticNet": ElasticNet(alpha=0.01, l1_ratio=0.5)
}.items():
    model.fit(Ztr, y_train.values)
    pred = model.predict(Zte)

    r2  = r2_score(y_test.values, pred)
    mse = mean_squared_error(y_test.values, pred)
    ic  = spearmanr(y_test.values, pred).correlation

    results_ae.append({"model": model_name, "OOS_R2": r2, "MSE": mse, "IC": ic})

results_ae = pd.DataFrame(results_ae)
display(results_ae)


Latent train shape: (199500, 8) | Latent test shape: (49900, 8)


,model,OOS_R2,MSE,IC
0,Ridge,0.005861,0.007371,0.043529
1,ElasticNet,0.007876,0.007356,0.045934


In [29]:
# %% [code] A1) Rank features by training IC IR

# feature_report must be computed ONLY on training data
# (you already did this correctly)

feature_ranked = (
    feature_report
    .sort_values("score", ascending=False)
    .reset_index(drop=True)
)

display(feature_ranked.head(15))
print("Total candidate features:", len(feature_ranked))


NameError: name 'feature_report' is not defined

In [30]:
# ============================================
# Feature subset sweep (k=5..50) using ranked features
# Models: Ridge + ElasticNet
# Data source folder: feature_engineering_outputs
# ============================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# ---------------------------
# 0) PATHS (your directory)
# ---------------------------
BASE_DIR = r"D:\Thesis\Old\Codes\New\feature_engineering_outputs"

RANK_FILE    = os.path.join(BASE_DIR, "feature_quality_report_10y_100tickers.csv")
FEATURES_PQ  = os.path.join(BASE_DIR, "features_10y_100tickers_wide.parquet")
TARGETS_PQ   = os.path.join(BASE_DIR, "targets_10y_100tickers_wide.parquet")

# ---------------------------
# 1) SETTINGS
# ---------------------------
K_LIST = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
N_SPLITS = 5

# fixed hyperparams (keep fixed so comparison is about k, not tuning)
RIDGE_ALPHA = 10.0
ENET_ALPHA = 0.01
ENET_L1_RATIO = 0.5
ENET_MAX_ITER = 20000

# If your target column name is known, set it here.
# Otherwise, the code will auto-pick a reasonable target column from targets parquet.
TARGET_COL = None  # e.g., "target_return" or "ret_fwd_21d" etc.

# ---------------------------
# 2) LOAD DATA
# ---------------------------
print("Loading rank file:", RANK_FILE)
rank_df = pd.read_csv(RANK_FILE)

print("Loading features:", FEATURES_PQ)
X_all = pd.read_parquet(FEATURES_PQ)

print("Loading targets:", TARGETS_PQ)
y_all_df = pd.read_parquet(TARGETS_PQ)

print("✅ Loaded shapes:")
print("   rank_df:", rank_df.shape)
print("   X_all  :", X_all.shape)
print("   y_all  :", y_all_df.shape)

# ---------------------------
# 3) PICK TARGET COLUMN
# ---------------------------
if TARGET_COL is None:
    # preferred names first
    preferred = ["target_return", "y", "target", "ret_fwd", "fwd_return", "next_return"]
    found = None
    for p in preferred:
        if p in y_all_df.columns:
            found = p
            break
    TARGET_COL = found if found is not None else y_all_df.columns[0]

print(f"🎯 Using target column: {TARGET_COL}")

y_all = y_all_df[TARGET_COL].copy()

# ---------------------------
# 4) GET RANKED FEATURE LIST
# ---------------------------
# Your rank file example had columns like: feature, score, ...
# We will use 'score' descending (higher = better)
if "feature" not in rank_df.columns:
    raise ValueError("❌ rank_df must contain a 'feature' column.")
if "score" not in rank_df.columns:
    raise ValueError("❌ rank_df must contain a 'score' column (your final ranking score).")

rank_df = rank_df.sort_values("score", ascending=False).copy()
ranked_features = rank_df["feature"].astype(str).tolist()

# Keep only features that exist in X_all
ranked_features = [f for f in ranked_features if f in X_all.columns]
if len(ranked_features) == 0:
    raise ValueError("❌ None of the ranked features exist in features parquet columns. Check naming.")

# cap K_LIST if needed
K_LIST = [k for k in K_LIST if k <= len(ranked_features)]
if len(K_LIST) == 0:
    raise ValueError("❌ K_LIST values exceed available ranked features. Reduce K_LIST or fix ranking/features mismatch.")

print(f"✅ Ranked features available in X_all: {len(ranked_features)}")
print("Top 10 ranked features found:", ranked_features[:10])

# ---------------------------
# 5) ALIGN X AND y (same index)
# ---------------------------
# Ensure same index
common_index = X_all.index.intersection(y_all.index)
X_all = X_all.loc[common_index]
y_all = y_all.loc[common_index]

# Sort by index for time series CV (important)
try:
    X_all = X_all.sort_index()
    y_all = y_all.sort_index()
except Exception:
    pass

# Drop rows with missing target
mask = y_all.notna()
X_all = X_all.loc[mask]
y_all = y_all.loc[mask]

# ---------------------------
# 6) CV EVALUATION FUNCTION
# ---------------------------
def evaluate_model_cv(X: pd.DataFrame, y: pd.Series, pipe: Pipeline, n_splits: int = 5):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    metrics = []

    Xv = X.values
    yv = y.values

    for fold, (tr, te) in enumerate(tscv.split(Xv), start=1):
        X_tr, X_te = Xv[tr], Xv[te]
        y_tr, y_te = yv[tr], yv[te]

        pipe.fit(X_tr, y_tr)
        pred = pipe.predict(X_te)

        r2 = r2_score(y_te, pred)
        mae = mean_absolute_error(y_te, pred)
        rmse = np.sqrt(mean_squared_error(y_te, pred))

        metrics.append({"fold": fold, "r2": r2, "mae": mae, "rmse": rmse})

    mdf = pd.DataFrame(metrics)
    summary = {
        "r2_mean": mdf["r2"].mean(),
        "r2_std":  mdf["r2"].std(ddof=1),
        "mae_mean": mdf["mae"].mean(),
        "mae_std":  mdf["mae"].std(ddof=1),
        "rmse_mean": mdf["rmse"].mean(),
        "rmse_std":  mdf["rmse"].std(ddof=1),
    }
    return mdf, summary

# ---------------------------
# 7) MODEL PIPELINES (scale inside CV to avoid leakage)
# ---------------------------
ridge_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=RIDGE_ALPHA, random_state=42))
])

enet_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", ElasticNet(alpha=ENET_ALPHA, l1_ratio=ENET_L1_RATIO, max_iter=ENET_MAX_ITER, random_state=42))
])

# ---------------------------
# 8) SWEEP OVER K
# ---------------------------
rows = []

for k in K_LIST:
    feats_k = ranked_features[:k]
    X_k = X_all[feats_k]

    # Drop rows with missing features for this subset (simple & consistent)
    valid = X_k.notna().all(axis=1)
    X_k2 = X_k.loc[valid]
    y_k2 = y_all.loc[valid]

    # if too few samples, skip
    if len(X_k2) < (N_SPLITS + 1) * 10:
        print(f"⚠️ Skipping k={k}: too few samples after NA drop ({len(X_k2)})")
        continue

    # Ridge
    _, ridge_sum = evaluate_model_cv(X_k2, y_k2, ridge_pipe, n_splits=N_SPLITS)
    rows.append({"model": "Ridge", "k": k, **ridge_sum})

    # ElasticNet
    _, enet_sum = evaluate_model_cv(X_k2, y_k2, enet_pipe, n_splits=N_SPLITS)
    rows.append({"model": "ElasticNet", "k": k, **enet_sum})

results = pd.DataFrame(rows).sort_values(["model", "k"]).reset_index(drop=True)

print("\n✅ RESULTS:")
display(results)

# ---------------------------
# 9) BEST K REPORTING
# ---------------------------
def best_k(df, model, metric, higher_is_better=True):
    tmp = df[df["model"] == model].copy()
    if tmp.empty:
        return None
    idx = tmp[metric].idxmax() if higher_is_better else tmp[metric].idxmin()
    return tmp.loc[idx, ["model", "k", metric]]

best_r2 = pd.DataFrame([
    best_k(results, "Ridge", "r2_mean", True),
    best_k(results, "ElasticNet", "r2_mean", True)
]).dropna()

best_rmse = pd.DataFrame([
    best_k(results, "Ridge", "rmse_mean", False),
    best_k(results, "ElasticNet", "rmse_mean", False)
]).dropna()

best_mae = pd.DataFrame([
    best_k(results, "Ridge", "mae_mean", False),
    best_k(results, "ElasticNet", "mae_mean", False)
]).dropna()

print("\n🏆 Best k by R² (higher better):")
display(best_r2)

print("\n🏆 Best k by RMSE (lower better):")
display(best_rmse)

print("\n🏆 Best k by MAE (lower better):")
display(best_mae)

# ---------------------------
# 10) SAVE OUTPUTS
# ---------------------------
out_csv = os.path.join(BASE_DIR, "k_sweep_ridge_elasticnet_results.csv")
results.to_csv(out_csv, index=False)
print(f"\n📁 Saved results to: {out_csv}")


Loading rank file: D:\Thesis\Old\Codes\New\feature_engineering_outputs\feature_quality_report_10y_100tickers.csv
Loading features: D:\Thesis\Old\Codes\New\feature_engineering_outputs\features_10y_100tickers_wide.parquet
Loading targets: D:\Thesis\Old\Codes\New\feature_engineering_outputs\targets_10y_100tickers_wide.parquet
✅ Loaded shapes:
   rank_df: (50, 13)
   X_all  : (2515, 5100)
   y_all  : (2515, 200)
🎯 Using target column: target_return_next1m__AAPL


ValueError: ❌ None of the ranked features exist in features parquet columns. Check naming.